# Separate Image Enhancer CNNs

Trains three independent CNN regressors on the generated split dataset. Each model predicts one correction parameter: brightness, contrast, or saturation.

In [ ]:
import csv
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    RESAMPLE_BILINEAR = Image.Resampling.BILINEAR
except AttributeError:
    RESAMPLE_BILINEAR = Image.BILINEAR

SEED = 42
IMAGE_SIZE = 384
BATCH_SIZE = 32
EPOCHS = 12
LR = 1e-3
NUM_WORKERS = 0
TARGET_NAMES = ["brightness", "contrast", "saturation"]

ROOT = Path.cwd()
if ROOT.name != "ml" and (ROOT / "ml").exists():
    ROOT = ROOT / "ml"

DATA_DIR = ROOT / "data" / "processed"
SPLIT_DIRS = {split: DATA_DIR / split for split in ("train", "val", "test")}
LABELS_CSV = DATA_DIR / "labels.csv"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")
print(f"Dataset: {DATA_DIR}")

In [ ]:
class SingleParameterDataset(Dataset):
    def __init__(self, split, target_name, labels_csv=LABELS_CSV, split_dirs=SPLIT_DIRS):
        if target_name not in TARGET_NAMES:
            raise ValueError(f"Unknown target {target_name}; expected one of {TARGET_NAMES}")
        self.split = split
        self.target_name = target_name
        self.split_dirs = {name: Path(path) for name, path in split_dirs.items()}
        with Path(labels_csv).open("r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            required = ["image_id", "brightness", "contrast", "saturation"]
            if reader.fieldnames is None or any(field not in reader.fieldnames for field in required):
                raise ValueError(f"Labels must contain {required}, got {reader.fieldnames}")
            self.rows = [row for row in reader if row["image_id"].startswith(f"{split}_")]

        if not self.rows:
            raise ValueError(f"No {split} labels found in {labels_csv}")
        missing = [row["image_id"] for row in self.rows if not (self.split_dirs[self.split] / f"{row['image_id']}.jpg").exists()]
        if missing:
            raise FileNotFoundError(f"Missing {len(missing)} {split} images, first={missing[0]}")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image_path = self.split_dirs[self.split] / f"{row['image_id']}.jpg"
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            if image.size != (IMAGE_SIZE, IMAGE_SIZE):
                image = image.resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE_BILINEAR)
            array = np.asarray(image, dtype=np.float32) / 255.0

        tensor = torch.from_numpy(array).permute(2, 0, 1)
        target = torch.tensor([float(row[self.target_name])], dtype=torch.float32)
        return tensor, target


def make_loaders(target_name):
    train_dataset = SingleParameterDataset("train", target_name)
    val_dataset = SingleParameterDataset("val", target_name)
    test_dataset = SingleParameterDataset("test", target_name)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    return train_loader, val_loader, test_loader


probe_loader, _, _ = make_loaders("brightness")
print(f"Samples: train={len(probe_loader.dataset)}")
next(iter(probe_loader))[0].shape, next(iter(probe_loader))[1].shape

In [ ]:
class SingleParameterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 96, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Linear(96, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
        )

    def forward(self, image):
        return self.head(self.features(image))


SingleParameterCNN().to(DEVICE)(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)).shape

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.SmoothL1Loss()
    total_loss = 0.0
    total_mae = 0.0
    count = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for image, target in loader:
            image = image.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            pred = model(image)
            loss = loss_fn(pred, target)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            batch_size = image.size(0)
            total_loss += loss.item() * batch_size
            total_mae += torch.abs(pred.detach() - target).sum().item()
            count += batch_size

    return total_loss / count, total_mae / count


def train_single_model(target_name, epochs=EPOCHS, lr=LR):
    torch.manual_seed(SEED)
    train_loader, val_loader, test_loader = make_loaders(target_name)
    model = SingleParameterCNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_state = None
    best_val = float("inf")

    for epoch in tqdm(range(epochs), desc=f"Training {target_name}"):
        train_loss, train_mae = run_epoch(model, train_loader, optimizer)
        val_loss, val_mae = run_epoch(model, val_loader)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        tqdm.write(
            f"{target_name} epoch={epoch + 1:02d} "
            f"train_loss={train_loss:.5f} val_loss={val_loss:.5f} "
            f"val_mae={val_mae:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)
    test_loss, test_mae = run_epoch(model, test_loader)
    print(f"{target_name} test_loss={test_loss:.5f} test_mae={test_mae:.4f}")
    return model, {"target": target_name, "best_val_loss": best_val, "test_loss": test_loss, "test_mae": test_mae}


In [ ]:
class CombinedSeparateEnhancer(nn.Module):
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleDict({name: models[name] for name in TARGET_NAMES})

    def forward(self, image):
        outputs = [self.models[name](image) for name in TARGET_NAMES]
        return torch.cat(outputs, dim=1)


def export_onnx(model, path):
    model.eval().cpu()
    dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
    torch.onnx.export(
        model,
        dummy,
        path,
        input_names=["image"],
        output_names=["params"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )
    return path


trained_models = {}
metrics = []
for target_name in TARGET_NAMES:
    model, result = train_single_model(target_name)
    trained_models[target_name] = model.cpu()
    metrics.append(result)

combined_model = CombinedSeparateEnhancer(trained_models)
checkpoint_path = MODEL_DIR / "enhancer_separate_combined.onnx"
export_onnx(combined_model, checkpoint_path)
print(f"Exported combined model: {checkpoint_path}")
metrics